# V10.1: Text as Supervision (R-Super Inspired)

**Paradigm shift:** Text is NOT a model input — it's a LOSS constraint.

Instead of feeding text into fusion layers (which the backbone ignores),
parse text to extract expected tumor properties and penalize the model
if its segmentation contradicts what the text says.

Text shapes the loss landscape, not the feature space.
This completely bypasses modality competition.

Reference: R-Super (MICCAI 2025 Best Paper)


In [ ]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi 2>/dev/null || echo 'No GPU'
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0: break
        if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else: raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'BraTS2020: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    lc = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(lc): return
    for f in glob.glob(os.path.join(lc, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    b = os.path.join(lc, 'best.pth')
    if os.path.exists(b): shutil.copy2(b, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
    l = os.path.join(lc, 'last.pth')
    if os.path.exists(l): shutil.copy2(l, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

_ss = threading.Event(); _st = {}
def _fs(p, w=3):
    try: s=os.path.getsize(p); time.sleep(w); return os.path.getsize(p)==s and s>0
    except: return False
def _bg():
    lc=os.path.join(REPO_DIR,'checkpoints')
    while not _ss.is_set():
        _ss.wait(120)
        if _ss.is_set(): break
        try:
            n=0
            for f in glob.glob(os.path.join(lc,'*.pth')):
                mt=os.path.getmtime(f); nm=os.path.basename(f)
                if nm not in _st or _st[nm]<mt:
                    if not _fs(f): continue
                    shutil.copy2(f,os.path.join(DRIVE_CKPT,nm)); _st[nm]=mt; n+=1
            if n>0: print(f'[AutoSync] {n} synced')
        except Exception as e: print(f'[AutoSync] {e}')
threading.Thread(target=_bg, daemon=True).start()
print('Setup complete')


## Training


In [ ]:
import os, glob
os.chdir(REPO_DIR)
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'best_V8.0_stage1.pth')
assert os.path.exists(STAGE1_CKPT), f'Not found: {STAGE1_CKPT}'

ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)
for f in glob.glob(os.path.join(ckpt_dir, '*.pth')):
    os.remove(f)

print(f'V10.1: Text Supervision from {STAGE1_CKPT}')
!python -u train.py \
    --config configs/autoresearch/V10.1_text_supervision.yaml \
    --resume "{STAGE1_CKPT}" \
    --reset-optimizer \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('V10.1')
print('V10.1 complete!')


In [ ]:
# === Emergency Sync ===
import shutil, glob, os, subprocess
lc = os.path.join(REPO_DIR, 'checkpoints')
fs = glob.glob(os.path.join(lc, '*.pth'))
if fs:
    for f in sorted(fs): shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    subprocess.run(['sync'], check=True)
    print(f'{len(fs)} files synced')
else: print('No checkpoints')


## Evaluation


In [ ]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V10.1.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V10.1_text_supervision.yaml'
for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG, '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout)
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-500:]}')

print()
print('Comparison:')
print('  V8.0 (pretrained):  Mean=0.8753, delta=0.00%')
print('  V10.0 (noise+film): Mean=0.8710, delta=-0.02%')
print('  V10.1 (text supervision): Mean=?, delta=?')
